# 10 · Critique and redesign

**Session 10 — Critique and Redesign** · about 40 minutes

**You will learn to**
- critique a figure with a shared vocabulary (ten simple rules)
- redesign misleading or confusing plots
- choose colours that work for everyone
- use an AI assistant as a critic while you stay the editor

**Data:** COVID-19 doses per day (Transparència Catalunya); synthetic exam scores.

**Outline**
0. Before you start (no AI)
1. Ten simple rules
2. Bad vs better: daily doses
3. Redesign a misleading chart
4. Colour
5. AI as critic, you as editor
6. Before/after for S12

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

In [ ]:
rng = np.random.default_rng(7)
school_type = np.repeat(["Public", "Private", "Concerted"], [120, 40, 70])
scores = np.concatenate([rng.normal(97.1, 5, 120), rng.normal(98.4, 5, 40), rng.normal(97.8, 5, 70)])
exams = pd.DataFrame({"school_type": school_type, "score": scores.clip(0, 110)})   # synthetic data

means = exams.groupby("school_type", sort=False)["score"].agg(["mean", "sem"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(means.index, means["mean"], yerr=1.96 * means["sem"], color=["red", "green", "orange"],
       capsize=6, label=means.index)
ax.set_ylim(96, 99.5)
ax.legend(means.index, loc="upper left")
ax.set_title("RESULTS!!!")
ax.grid(True, color="black")
plt.show()

**Three minutes, alone, on paper:** list three problems in this chart. Then compare in pairs.
Only after that, ask an AI assistant the same question (section 5).

## 1. Ten simple rules for better figures

Rougier, Droettboom & Bourne (2014), *PLOS Computational Biology* 10(9): e1003833,
[doi:10.1371/journal.pcbi.1003833](https://doi.org/10.1371/journal.pcbi.1003833):

1. Know your audience · 2. Identify your message · 3. Adapt the figure to the support medium ·
4. Captions are not optional · 5. Do not trust the defaults · 6. Use colour effectively ·
7. Do not mislead the reader · 8. Avoid chartjunk · 9. Message trumps beauty · 10. Get the right tool

Quote the rule number when you give feedback: *“rule 7: the axis is truncated”*.

## 2. Bad vs better: daily doses

A line through daily counts (last year's example) is dominated by the weekly cycle:

In [ ]:
raw = pd.read_parquet(DATA / "examples/covid_vaccination_daily.parquet")
raw["date"] = pd.to_datetime(raw["data"], format="ISO8601")
daily = raw.groupby("date")["dosis"].sum()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(daily.index, daily)
ax.set_title("Vaccines per day in Catalonia")
plt.show()

### TASK I — a better version

1. Aggregate to **weekly** totals (`weekly`).
2. Draw bars with a **log** y-axis (`ax.set_yscale("log")`).
3. Highlight in a second colour the weeks above 300,000 doses.
4. Bold, larger axis labels with units; a title that states the message.

In [ ]:
# TASK I — your code here
weekly, ax = None, None

In [ ]:
# ✅ Self-check TASK I
assert weekly is not None and weekly.index.freqstr.startswith("W")
assert ax.get_yscale() == "log", "Use a log y-axis."
assert ax.get_xlabel() and ax.get_ylabel()
assert len({p.get_facecolor() for p in ax.patches}) == 2, "Two colours: normal and highlighted weeks."
print("✅ TASK I passed")

## 3. Redesign a misleading chart

### TASK II — redesign the exam chart from section 0

Using `exams`, make `fig2, ax2` so that:

1. the y-axis starts at 0 **or** you show the individual scores (points / strip plot), so the size of the
   differences is honest (rule 7);
2. it uses one neutral colour or a colour-blind-safe palette — no red vs green (rule 6);
3. no legend repeats the x labels; the y-axis has a label with units;
4. the title states the message.

*Sketch it first.*

In [ ]:
# TASK II — your code here
fig2, ax2 = None, None

In [ ]:
# ✅ Self-check TASK II
assert ax2 is not None and ax2.get_ylabel(), "Label the y-axis with units."
shows_points = any(len(c.get_offsets()) > 50 for c in ax2.collections)
assert ax2.get_ylim()[0] <= 0 or shows_points, "Start at zero or show the individual scores."
assert ax2.get_legend() is None, "No legend that repeats the x labels."
print("✅ TASK II passed")

## 4. Colour

- **Amounts** → a perceptually uniform sequential map (`viridis`, `cividis`).
- **Values around a meaningful centre** → a diverging map (`RdBu`) centred on that value.
- **Categories** → at most about seven distinct colours; label lines directly instead of a legend.
- About 1 in 12 men has red–green colour-vision deficiency: never rely on red vs green alone.

[Choosing colormaps in Matplotlib](https://matplotlib.org/stable/users/explain/colors/colormaps.html) ·
[ColorBrewer](https://colorbrewer2.org/)

In [ ]:
gradient = np.linspace(0, 1, 256)[None, :]
fig, axs = plt.subplots(4, 1, figsize=(7, 2.6), layout="constrained")
for ax, (name, note) in zip(axs, [("viridis", "sequential"), ("cividis", "sequential, CVD-friendly"),
                                  ("RdBu", "diverging"), ("jet", "avoid")]):
    ax.imshow(gradient, aspect="auto", cmap=name)
    ax.set_axis_off()
    ax.set_title(f"{name} — {note}", loc="left", fontsize=9)
plt.show()

## 5. AI as critic, you as editor

1. Give an AI assistant **one** of your figures (as an image) and its intended message. Ask for the five most
   important problems.
2. Decide on each point and record it below (copy the table into the figure's GitHub issue).
3. Compare with your peers' feedback: where do they agree? What did the AI miss (context, audience, your data)?

| # | AI's point | Accept / reject | Why |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

The AI does not know your data or your audience: **you decide** what changes.

## 6. Before/after for Session 12

Export the before and after versions of your strongest figure for the plot presentations:

In [ ]:
if fig2 is not None:
    fig2.savefig(OUTPUTS / "exam_scores_after.png", dpi=200, bbox_inches="tight")
fig.savefig(OUTPUTS / "colormaps.png", dpi=200, bbox_inches="tight")
sorted(p.name for p in OUTPUTS.glob("*.png"))

Take-away checklist for your final figures:
[Friends Don't Let Friends Make Bad Graphs](https://github.com/cxli233/FriendsDontLetFriends) (Chenxin Li).